# **`Analiza osjetljivosti hiperparametara`**
### *Kvantifikacija nesigurnosti u modelima umjetne inteligencije - okvir za prediktivno održavanje i analizu rizika*



## 1. Učitavanje biblioteka


> *Importovanje potrebnih biblioteka za rad.*


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error
import random

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

## 2. Učitavanje seta podataka (*CMAPSS* )

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
train = pd.read_csv('/content/drive/MyDrive/CMAPSS/train_FD001.txt', sep=r'\s+', header=None)

test = pd.read_csv('/content/drive/MyDrive/CMAPSS/test_FD001.txt', sep=r'\s+', header=None)

rul = pd.read_csv('/content/drive/MyDrive/CMAPSS/RUL_FD001.txt', sep=r'\s+', header=None)

*Napomena: U ovom radu koristi se podskup FD001 koji sadrži jedan operativni uvjet i jedan tip kvara. Ovaj podskup je odabran jer omogućava jednostavniju i jasniju analizu metoda za kvantifikaciju nesigurnosti, bez dodatne složenosti koju donose različiti režimi rada i više tipova kvarova.*

In [ ]:
train.head()

,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,25
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


## 3. Eksploratorna analiza podataka (*EDA* )



> *Prije preprocesiranja podataka izvršena je eksploratorna analiza podataka (EDA) s ciljem boljeg razumijevanja strukture CMAPSS FD001 dataseta, ponašanja senzora i obrazaca degradacije motora. Analiza uključuje pregled osnovnih statistika, distribucije životnog vijeka motora i RUL vrijednosti, kao i identifikaciju senzora koji nose korisnu informaciju za predikciju. Rezultati ove analize korišteni su za donošenje odluka u fazi preprocesiranja i odabira relevantnih ulaznih karakteristika za modele.*



In [ ]:
columns = ['engine_id', 'cycle']

operational_settings = [f'op_setting_{i}' for i in range(1, 4)] # Postavljanje naziva kolona (operativni + senzorski podaci)
sensor_columns = [f'sensor_{i}' for i in range(1, 22)]

columns += operational_settings + sensor_columns

train.columns = columns
test.columns = columns

## 4. Priprema i obrada podataka



> *Na osnovu EDA identificirani su konstantni senzori koji ne nose informaciju o degradaciji te će biti isključeni iz feature seta. U ovoj sekciji definišemo konačni skup varijabli, računamo ciljnu varijablu RUL, normalizujemo podatke i kreiramo vremenske sekvence pogodne za LSTM arhitekturu.*




In [ ]:
train.head()

,engine_id,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [ ]:
# Uklanjamo konstantne senzore identificirane u EDA
constant_sensors = ['sensor_1', 'sensor_5', 'sensor_6',
                    'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']

feature_columns = [col for col in operational_settings + sensor_columns
                   if col not in constant_sensors]

print(f"Originalni broj feattura: {len(operational_settings + sensor_columns)}")
print(f"Nakon uklanjanja konstantnih senzora: {len(feature_columns)}")
print(f"Zadržani featuri: {feature_columns}")

Originalni broj feattura: 24
Nakon uklanjanja konstantnih senzora: 17
Zadržani featuri: ['op_setting_1', 'op_setting_2', 'op_setting_3', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']


Feature set je reduciran sa 24 na 17 varijabli. Tri operativna parametra zadržana su jer FD001 koristi jedan operativni uvjet - njihova varijansa je minimalna ali ih zadržavamo radi konzistentnosti. Eliminacijom sedam konstantnih senzora smanjujemo dimenzionalnost ulaza bez gubitka informacije.

U okviru predprocesiranja definiše se ciljna varijabla Remaining Useful Life (RUL), koja označava preostali broj ciklusa rada do otkaza motora. Vrijednost RUL-a dobija se kao razlika između maksimalnog broja ciklusa i trenutnog ciklusa za svaki pojedinačni motor.

In [ ]:
train['RUL'] = train.groupby('engine_id')['cycle'].transform('max') - train['cycle']

RUL_MAX = 125
train['RUL'] = train['RUL'].clip(upper=RUL_MAX)

print(train[['engine_id', 'cycle', 'RUL']].head(3))
print(f"\nRaspon RUL-a nakon clippinga: [{train['RUL'].min()}, {train['RUL'].max()}]")

   engine_id  cycle  RUL
0          1      1  125
1          1      2  125
2          1      3  125

Raspon RUL-a nakon clippinga: [0, 125]


Clipping RUL-a na 125 ciklusa zasniva se na pretpostavci da motor u ranim fazama rada ne pokazuje mjerljive znakove degradacije. Ova "piecewise linear" pretpostavka je u skladu sa literaturom o CMAPSS setu podataka (Saxena et al., 2008) i potvrđena je vizualno u EDA sekciji kroz distribuciju životnog vijeka motora.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

train[feature_columns] = scaler.fit_transform(train[feature_columns])
test[feature_columns] = scaler.transform(test[feature_columns])

MinMaxScaler skalira sve feature na interval [0, 1]. Scaler se fituje isključivo na trening setu, a isti parametri se primjenjuju na test set - čime se sprečava curenje podataka.

In [ ]:
seq_length = 30

def create_sequences(data, seq_length, feature_columns):
    xs, ys = [], []
    for engine_id in data['engine_id'].unique():
        engine_data = data[data['engine_id'] == engine_id]
        for i in range(len(engine_data) - seq_length):
            x = engine_data.iloc[i:i+seq_length][feature_columns].values
            y = engine_data.iloc[i+seq_length]['RUL']
            xs.append(x)
            ys.append(y)
    return np.array(xs), np.array(ys)

X_train, y_train = create_sequences(train, seq_length, feature_columns)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"\nTumačenje: {X_train.shape[0]} uzoraka, {X_train.shape[1]} vremenskih koraka, {X_train.shape[2]} featura")

X_train shape: (17631, 30, 17)
y_train shape: (17631,)

Tumačenje: 17631 uzoraka, 30 vremenskih koraka, 17 featura


Svaka sekvenca predstavlja prozor od 30 uzastopnih ciklusa jednog motora, a ciljna vrijednost je RUL na kraju tog prozora.

In [ ]:
X_test = []

for engine_id in test['engine_id'].unique():
    engine_data = test[test['engine_id'] == engine_id]
    if len(engine_data) >= seq_length:
        seq = engine_data.iloc[-seq_length:][feature_columns].values
        X_test.append(seq)

X_test = np.array(X_test)
y_test = rul[0].values

print(f"X_test shape:  {X_test.shape}")
print(f"y_test shape:  {y_test.shape}")
print(f"Broj motora u testu: {len(y_test)}")

X_test shape:  (100, 30, 17)
y_test shape:  (100,)
Broj motora u testu: 100


Za testne motore uzima se samo posljednjih 30 ciklusa - to odgovara scenariju gdje u realnoj primjeni imamo historijat do trenutnog stanja motora i želimo predvidjeti koliko mu je preostalo.

In [ ]:
from sklearn.model_selection import train_test_split

# Grupni split po motoru (engine_id) - sprječava curenje sekvenci istog motora između trening i validacionog skupa
np.random.seed(42)

all_engine_ids = train['engine_id'].unique()
train_ids, val_ids = train_test_split(all_engine_ids, test_size=0.2, random_state=42)

train_subset = train[train['engine_id'].isin(train_ids)]
val_subset   = train[train['engine_id'].isin(val_ids)]

X_tr, y_tr   = create_sequences(train_subset, seq_length, feature_columns)
X_val, y_val = create_sequences(val_subset,   seq_length, feature_columns)

print(f"Trening motora: {len(train_ids)}, validacionih motora: {len(val_ids)}")
print(f"X_tr shape: {X_tr.shape}, X_val shape: {X_val.shape}")

Trening motora: 80, validacionih motora: 20
X_tr shape: (14161, 30, 17), X_val shape: (3470, 30, 17)


### DenseVariational klasa za BNN model

Ovaj kod za BNN model je potreban kako bi se mogla sprovesti analiza osjetljivosti hiperparametara kod tog modela.

In [ ]:
import random
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
# BNN — Custom DenseVariational sloj (reparametrization trick)
n_train = X_tr.shape[0]

class DenseVariational(tf.keras.layers.Layer):
    def __init__(self, units, activation=None, kl_weight=1.0, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.activation = tf.keras.activations.get(activation)
        self.kl_weight = kl_weight

    def build(self, input_shape):
        n_inputs = int(input_shape[-1])

        # Parametri distribucije kernela
        self.kernel_mu = self.add_weight(
            name='kernel_mu',
            shape=(n_inputs, self.units),
            initializer='glorot_normal',
            trainable=True
        )
        self.kernel_rho = self.add_weight(
            name='kernel_rho',
            shape=(n_inputs, self.units),
            initializer=tf.initializers.constant(-3.0),
            trainable=True
        )

        # Parametri distribucije biasa
        self.bias_mu = self.add_weight(
            name='bias_mu',
            shape=(self.units,),
            initializer='zeros',
            trainable=True
        )
        self.bias_rho = self.add_weight(
            name='bias_rho',
            shape=(self.units,),
            initializer=tf.initializers.constant(-3.0),
            trainable=True
        )

    def call(self, inputs, training=None):
        kernel_sigma = tf.nn.softplus(self.kernel_rho) + 1e-5
        bias_sigma   = tf.nn.softplus(self.bias_rho)   + 1e-5

        if training:
            # Reparametrization trick: w = mu + sigma * epsilon
            kernel = self.kernel_mu + kernel_sigma * tf.random.normal(self.kernel_mu.shape)
            bias   = self.bias_mu   + bias_sigma   * tf.random.normal(self.bias_mu.shape)
        else:
            kernel = self.kernel_mu
            bias   = self.bias_mu

        # KL divergencija prema standardnom normalu N(0,1)
        kl = self._kl_divergence(self.kernel_mu, kernel_sigma) + \
             self._kl_divergence(self.bias_mu,   bias_sigma)

        self.add_loss(self.kl_weight * kl / n_train)

        output = tf.matmul(inputs, kernel) + bias
        return self.activation(output) if self.activation else output

    def _kl_divergence(self, mu, sigma):
        return 0.5 * tf.reduce_sum(
            tf.square(mu) + tf.square(sigma) - tf.math.log(tf.square(sigma)) - 1.0
        )


def build_bnn_model(seq_length, n_features, n_train):
    kl_weight = 1.0

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(seq_length, n_features)),

        # LSTM slojevi — deterministički
        tf.keras.layers.LSTM(64, return_sequences=True),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.LSTM(32),
        tf.keras.layers.Dropout(0.2),

        # Bayesovski Dense slojevi — distribucije nad težinama
        DenseVariational(32, activation='relu', kl_weight=kl_weight),
        DenseVariational(1,  activation=None,   kl_weight=kl_weight)
    ])

    return model


bnn_model = build_bnn_model(seq_length, len(feature_columns), n_train)
bnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 64)         │        20,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_variational               │ (None, 32)             │         2,112 │
│ (DenseVariational)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_variational_1             │ (None, 1)              │            66 │
│ (DenseVariational)              │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 35,586 (139.01 KB)

 Trainable params: 35,586 (139.01 KB)

 Non-trainable params: 0 (0.00 B)

## 5. Analiza osjetljivosti hiperparametara

Umjesto opravdanja izbora hiperparametara isključivo preliminarnim probama, ovdje se provodi one-at-a-time (OFAT) analiza osjetljivosti na validacionom skupu (train_ids/val_ids, Sekcija 3). Za svaki hiperparametar testira se nekoliko vrijednosti dok se ostali drže fiksnim na finalnim vrijednostima. Svaki eksperiment ponavlja se sa 3 različita random seed-a (42, 123, 7) radi provjere da li su razlike između vrijednosti stvarne ili posljedica šuma u treningu.

Napomena o ograničenju metode: OFAT pristup ne hvata interakcije između
hiperparametara i ne garantuje globalno optimalnu kombinaciju. Korišten je zbog
obima rada i računskih ograničenja, umjesto pune multi-dimenzionalne optimizacije (grid/random/Bayesian search).

Kriterijum izbora: bira se vrijednost sa najnižim srednjim val RMSE; ako je razlika između kandidata unutar jedne standardne devijacije, prednost se daje jednostavnijoj ili računski jeftinijoj vrijednosti, odnosno vrijednosti konzistentnoj sa ostalim modelima u radu.

#### `5.1 Osjetljivost na veličinu prozora W`

Pored uticaja na tačnost, provjerava se i praktično ograničenje: veći W zahtijeva
duži historijat po motoru, pa testni motori sa malim brojem ciklusa mogu biti
isključeni iz realne primjene modela.

In [ ]:
SEEDS = [42, 123, 7]
W_VALUES = [15, 20, 25, 30, 40, 50, 60]
W_MAX = max(W_VALUES)

# Fer poređenje: samo motori koji imaju dovoljno ciklusa za NAJVEĆI testirani W
valid_engines_train = train_subset.groupby('engine_id')['cycle'].max()
valid_engines_train = valid_engines_train[valid_engines_train >= W_MAX].index
train_subset_fair = train_subset[train_subset['engine_id'].isin(valid_engines_train)]

valid_engines_val = val_subset.groupby('engine_id')['cycle'].max()
valid_engines_val = valid_engines_val[valid_engines_val >= W_MAX].index
val_subset_fair = val_subset[val_subset['engine_id'].isin(valid_engines_val)]

print(f"Fer poređenje na {len(valid_engines_train)} trening i "
      f"{len(valid_engines_val)} validacionih motora (od originalnih "
      f"{train_subset['engine_id'].nunique()} i {val_subset['engine_id'].nunique()})")

results_w = []
for W in W_VALUES:
    rmse_runs = []
    for seed in SEEDS:
        random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

        X_tr_w, y_tr_w   = create_sequences(train_subset_fair, W, feature_columns)
        X_val_w, y_val_w = create_sequences(val_subset_fair,   W, feature_columns)

        m_w = Sequential([
            Input(shape=(W, len(feature_columns))),
            LSTM(64, return_sequences=True), Dropout(0.3),
            LSTM(32), Dropout(0.3),
            Dense(32, activation='relu'), Dense(1)
        ])
        m_w.compile(optimizer='adam', loss='mse', metrics=['mae'])
        hist_w = m_w.fit(X_tr_w, y_tr_w, epochs=50, batch_size=64,
                          validation_data=(X_val_w, y_val_w),
                          callbacks=[EarlyStopping(monitor='val_loss', patience=10,
                                                   restore_best_weights=True)],
                          verbose=0)
        rmse_runs.append(np.sqrt(min(hist_w.history['val_loss'])))

    n_valid_test = (test.groupby('engine_id')['cycle'].max() >= W).sum()
    n_total_test = test['engine_id'].nunique()

    results_w.append({
        'W': W,
        'val_RMSE_mean': round(np.mean(rmse_runs), 4),
        'val_RMSE_std':  round(np.std(rmse_runs), 4),
        'test_motora_pokriveno': f"{n_valid_test}/{n_total_test}"
    })

df_sensitivity_w = pd.DataFrame(results_w)
print("\nSensitivity analiza - veličina prozora (W), fer poređenje na fiksnom podskupu motora:")
print(df_sensitivity_w.to_string(index=False))

Fer poređenje na 80 trening i 20 validacionih motora (od originalnih 80 i 20)

Sensitivity analiza - veličina prozora (W), fer poređenje na fiksnom podskupu motora:
 W  val_RMSE_mean  val_RMSE_std test_motora_pokriveno
15        15.0124        0.1817               100/100
20        14.0190        0.0762               100/100
25        12.9323        0.1950               100/100
30        12.1226        0.0499               100/100
40        10.9392        0.2794                96/100
50        10.8090        0.3149                93/100
60        10.6096        0.1240                88/100


#### `5.2 Osjetljivost na dropout stopu p (LSTM arhitektura)`

Pored tačnosti (RMSE), provjerava se i kalibracija nesigurnosti kroz MC Dropout
inferenciju (50 stohastičkih prolaza, 95% interval): coverage probability i
prosječna širina intervala.

In [ ]:
def evaluate_mc_dropout(model, X_eval, y_eval, n_samples=50, z=1.96):
    preds = np.array([model(X_eval, training=True).numpy().flatten()
                       for _ in range(n_samples)])
    mean_pred, std_pred = preds.mean(axis=0), preds.std(axis=0)
    lower, upper = mean_pred - z * std_pred, mean_pred + z * std_pred

    rmse     = np.sqrt(mean_squared_error(y_eval, mean_pred))
    coverage = np.mean((y_eval >= lower) & (y_eval <= upper))
    width    = np.mean(upper - lower)
    return rmse, coverage, width

P_VALUES = [0.1, 0.2, 0.3, 0.4, 0.5]

results_p = []
for p in P_VALUES:
    rmse_runs, cov_runs, width_runs = [], [], []
    for seed in SEEDS:
        random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

        m_p = Sequential([
            Input(shape=(seq_length, len(feature_columns))),
            LSTM(64, return_sequences=True), Dropout(p),
            LSTM(32), Dropout(p),
            Dense(32, activation='relu'), Dense(1)
        ])
        m_p.compile(optimizer='adam', loss='mse', metrics=['mae'])
        m_p.fit(X_tr, y_tr, epochs=50, batch_size=64,
                validation_data=(X_val, y_val),
                callbacks=[EarlyStopping(monitor='val_loss', patience=10,
                                         restore_best_weights=True)],
                verbose=0)

        random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
        rmse_p, cov_p, width_p = evaluate_mc_dropout(m_p, X_val, y_val)
        rmse_runs.append(rmse_p); cov_runs.append(cov_p); width_runs.append(width_p)

    results_p.append({
        'p': p,
        'val_RMSE_mean':       round(np.mean(rmse_runs), 4),
        'val_RMSE_std':        round(np.std(rmse_runs), 4),
        'val_Coverage_mean':   round(np.mean(cov_runs), 4),
        'val_Coverage_std':    round(np.std(cov_runs), 4),
        'mean_interval_width': round(np.mean(width_runs), 4),
        'interval_width_std':  round(np.std(width_runs), 4)
    })

df_sensitivity_p = pd.DataFrame(results_p)
print("Sensitivity analiza - dropout stopa p (LSTM grana, MC Dropout):")
print(df_sensitivity_p.to_string(index=False))

Sensitivity analiza - dropout stopa p (LSTM grana, MC Dropout):
  p  val_RMSE_mean  val_RMSE_std  val_Coverage_mean  val_Coverage_std  mean_interval_width  interval_width_std
0.1        12.0661        0.0949             0.5867            0.0153            19.160999              0.8055
0.2        12.3326        0.1270             0.7478            0.0250            28.318399              1.0583
0.3        12.0640        0.1669             0.7458            0.0387            28.473301              4.3929
0.4        12.2292        0.2101             0.7688            0.0217            32.454498              4.5888
0.5        12.4998        0.0564             0.7986            0.0379            35.096001              4.6566


#### `5.3 Osjetljivost na broj epoha`

Glavni dokaz je kriva konvergencije sa EarlyStopping-om (do 100 epoha) — pokazuje
na kojoj epohi val_loss prestaje padati. Dodatno, provjerava se val RMSE za
fiksan broj epoha (bez EarlyStopping-a) radi uvida u diminishing returns.

In [ ]:
random.seed(42); np.random.seed(42); tf.random.set_seed(42)

m_ep = Sequential([
    Input(shape=(seq_length, len(feature_columns))),
    LSTM(64, return_sequences=True), Dropout(0.3),
    LSTM(32), Dropout(0.3),
    Dense(32, activation='relu'), Dense(1)
])
m_ep.compile(optimizer='adam', loss='mse', metrics=['mae'])

es_ep = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
hist_ep = m_ep.fit(X_tr, y_tr, epochs=100, batch_size=64,
                    validation_data=(X_val, y_val),
                    callbacks=[es_ep], verbose=0)

best_epoch = np.argmin(hist_ep.history['val_loss']) + 1
print(f"Trening zaustavljen nakon {len(hist_ep.history['loss'])} epoha "
      f"(EarlyStopping patience=10).")
print(f"Najbolja epoha (min val_loss): {best_epoch}")
print(f"Val RMSE na najboljoj epohi: {np.sqrt(min(hist_ep.history['val_loss'])):.4f}")

Trening zaustavljen nakon 35 epoha (EarlyStopping patience=10).
Najbolja epoha (min val_loss): 25
Val RMSE na najboljoj epohi: 12.0522


In [ ]:
EPOCH_VALUES = [20, 50, 80]

results_epochs = []
for n_epochs in EPOCH_VALUES:
    rmse_runs = []
    for seed in SEEDS:
        random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

        m_e = Sequential([
            Input(shape=(seq_length, len(feature_columns))),
            LSTM(64, return_sequences=True), Dropout(0.3),
            LSTM(32), Dropout(0.3),
            Dense(32, activation='relu'), Dense(1)
        ])
        m_e.compile(optimizer='adam', loss='mse', metrics=['mae'])
        hist_e = m_e.fit(X_tr, y_tr, epochs=n_epochs, batch_size=64,
                          validation_data=(X_val, y_val), verbose=0)
        rmse_runs.append(np.sqrt(hist_e.history['val_loss'][-1]))

    results_epochs.append({
        'epochs': n_epochs,
        'val_RMSE_mean': round(np.mean(rmse_runs), 4),
        'val_RMSE_std':  round(np.std(rmse_runs), 4)
    })

df_sensitivity_epochs = pd.DataFrame(results_epochs)
print("Sensitivity analiza - broj epoha (bez EarlyStopping-a):")
print(df_sensitivity_epochs.to_string(index=False))

Sensitivity analiza - broj epoha (bez EarlyStopping-a):
 epochs  val_RMSE_mean  val_RMSE_std
     20        12.5035        0.0934
     50        12.9594        0.4030
     80        13.6943        0.4568


#### `5.4 Osjetljivost na dropout stopu p (BNN arhitektura)`

BNN već ima vlastiti izvor nesigurnosti kroz varijacione (DenseVariational)
slojeve, pa se provjerava da li dodatni dropout u LSTM granama treba biti manji
nego u čisto determinističkim granama (p=0.3 korišten kod MC Dropout/Ensemble).

In [ ]:
P_VALUES_BNN = [0.1, 0.2, 0.3, 0.4]

results_p_bnn = []
for p in P_VALUES_BNN:
    rmse_runs, cov_runs, width_runs = [], [], []
    for seed in SEEDS:
        random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

        m_bnn_p = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(seq_length, len(feature_columns))),
            tf.keras.layers.LSTM(64, return_sequences=True),
            tf.keras.layers.Dropout(p),
            tf.keras.layers.LSTM(32),
            tf.keras.layers.Dropout(p),
            DenseVariational(32, activation='relu', kl_weight=1.0),
            DenseVariational(1,  activation=None,   kl_weight=1.0)
        ])
        m_bnn_p.compile(optimizer='adam', loss='mse', metrics=['mae'])
        m_bnn_p.fit(X_tr, y_tr, epochs=50, batch_size=64,
                    validation_data=(X_val, y_val),
                    callbacks=[EarlyStopping(monitor='val_loss', patience=10,
                                             restore_best_weights=True)],
                    verbose=0)

        random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
        rmse_p, cov_p, width_p = evaluate_mc_dropout(m_bnn_p, X_val, y_val,
                                                      n_samples=100)
        rmse_runs.append(rmse_p); cov_runs.append(cov_p); width_runs.append(width_p)

    results_p_bnn.append({
        'p': p,
        'val_RMSE_mean':       round(np.mean(rmse_runs), 4),
        'val_RMSE_std':        round(np.std(rmse_runs), 4),
        'val_Coverage_mean':   round(np.mean(cov_runs), 4),
        'val_Coverage_std':    round(np.std(cov_runs), 4),
        'mean_interval_width': round(np.mean(width_runs), 4),
        'interval_width_std':  round(np.std(width_runs), 4)
    })

df_sensitivity_p_bnn = pd.DataFrame(results_p_bnn)
print("Sensitivity analiza - dropout stopa p (BNN arhitektura):")
print(df_sensitivity_p_bnn.to_string(index=False))

Sensitivity analiza - dropout stopa p (BNN arhitektura):
  p  val_RMSE_mean  val_RMSE_std  val_Coverage_mean  val_Coverage_std  mean_interval_width  interval_width_std
0.1        11.9221        0.1318             0.6497            0.0180            20.294500              1.3542
0.2        12.1732        0.3035             0.7252            0.0342            27.390499              1.2278
0.3        12.2723        0.3574             0.7823            0.0168            33.325298              0.9800
0.4        12.2525        0.2704             0.8094            0.0288            35.280499              4.0231
